# 07 — Full Validation Report

**Smart Road Assistant — Computer Vision Project**

This notebook consolidates all validation results across the two-stage cascade pipeline following standard CV project evaluation practices.

| Section | What is evaluated | Key metrics |
|---------|------------------|-------------|
| 1 | Dataset statistics | Split counts, class distribution, balance |
| 2 | Stage 1 Detector (test set) | mAP@0.5, mAP@0.5:0.95, PR curves, F1-conf curve, confusion matrix |
| 3 | Stage 2a Severity CNN (test set) | Accuracy, confusion matrix, per-class F1, ROC curves |
| 4 | Stage 2a Severity CV (test set) | Same metrics for classical baseline comparison |
| 5 | End-to-end pipeline | FPS, annotated examples, warning accuracy |
| 6 | Summary dashboard | All key metrics in one view |

> **Run order:** Run all cells top-to-bottom. Each section is self-contained.

---
## Section 0 — Imports & Paths

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path
from collections import Counter

import torch
from ultralytics import YOLO
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score,
    roc_curve, auc, precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

BASE_DIR      = Path('..').resolve()
MODELS_DIR    = BASE_DIR / 'models'
DATA_DIR      = BASE_DIR / 'data' / 'processed'
DETECTOR_DATA = DATA_DIR / 'detector_yolo'
SEVERITY_DATA = DATA_DIR / 'severity_crops'
RUNS_DIR      = BASE_DIR / 'runs'

DETECTOR_MODEL  = MODELS_DIR / 'detector_model.pt'
SEVERITY_MODEL  = MODELS_DIR / 'severity_model.pt'

DETECTOR_CLASSES = ['pothole', 'traffic_light']
SEVERITY_CLASSES = ['Low', 'Medium', 'High']
CNN_IDX_TO_LABEL = {0: 'High', 1: 'Low', 2: 'Medium'}  # YOLOv8-cls alphabetical order

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# GPU info
print('=' * 50)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU  : {p.name}  ({p.total_memory/1e9:.1f} GB VRAM)')
else:
    print('  Device: CPU (inference will be slower)')
print('=' * 50)

# Quick sanity check
missing = [p for p in [DETECTOR_MODEL, SEVERITY_MODEL] if not p.exists()]
if missing:
    print('\nMissing models:')
    for m in missing:
        print(f'  {m.name} — run the training notebook first')
else:
    print('\nAll models found.')

---
## Section 1 — Dataset Statistics

Understanding split composition and class balance before interpreting any metrics.

In [ ]:
# ── 1a. Detector dataset (YOLO format) ──────────────────────────────
def count_yolo_labels(split_dir):
    """Count bounding box instances per class from YOLO .txt labels."""
    counts = Counter()
    label_dir = split_dir.parent.parent / 'labels' / split_dir.name
    for txt in label_dir.glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    return counts

splits       = ['train', 'val', 'test']
img_counts   = {}
label_counts = {}

for sp in splits:
    img_dir = DETECTOR_DATA / 'images' / sp
    if img_dir.exists():
        imgs = list(img_dir.glob('*'))
        img_counts[sp] = len(imgs)
        label_counts[sp] = count_yolo_labels(img_dir)
    else:
        img_counts[sp] = 0
        label_counts[sp] = Counter()

print('Stage 1 Detector dataset')
print('-' * 50)
print(f'  {"Split":<8} {"Images":>8}  {"Pothole":>10}  {"Traffic":>10}')
print('-' * 50)
for sp in splits:
    lc = label_counts[sp]
    print(f'  {sp:<8} {img_counts[sp]:>8}  {lc[0]:>10}  {lc[1]:>10}')
total_imgs = sum(img_counts.values())
print('-' * 50)
print(f'  {"TOTAL":<8} {total_imgs:>8}')

In [ ]:
# ── 1b. Severity crop dataset (ImageFolder format) ──────────────────
sev_counts = {}
for sp in splits:
    sev_counts[sp] = {}
    for cls in SEVERITY_CLASSES:
        d = SEVERITY_DATA / sp / cls
        sev_counts[sp][cls] = len(list(d.glob('*'))) if d.exists() else 0

print('Stage 2a Severity crop dataset')
print('-' * 48)
print(f'  {"Split":<8} {"Low":>8}  {"Medium":>8}  {"High":>8}  {"Total":>8}')
print('-' * 48)
for sp in splits:
    row  = sev_counts[sp]
    tot  = sum(row.values())
    print(f'  {sp:<8} {row["Low"]:>8}  {row["Medium"]:>8}  {row["High"]:>8}  {tot:>8}')
print('-' * 48)

In [ ]:
# ── 1c. Visual — class distributions ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Detector image counts per split
ax = axes[0]
bars = ax.bar(splits, [img_counts[s] for s in splits],
              color=['#3498db', '#e67e22', '#2ecc71'], width=0.5)
for bar, v in zip(bars, [img_counts[s] for s in splits]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(v), ha='center', fontweight='bold')
ax.set_title('Detector: Images per Split', fontweight='bold')
ax.set_ylabel('Image count')
ax.grid(alpha=0.3, axis='y')

# Detector class distribution (stacked bar)
ax = axes[1]
pot = [label_counts[s][0] for s in splits]
tl  = [label_counts[s][1] for s in splits]
x   = np.arange(len(splits))
ax.bar(x, pot, label='pothole',       color='#e74c3c', width=0.5)
ax.bar(x, tl,  bottom=pot,           label='traffic_light', color='#2ecc71', width=0.5)
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_title('Detector: Instance Distribution', fontweight='bold')
ax.set_ylabel('Bounding box instances')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# Severity crop distribution (grouped bar)
ax = axes[2]
sev_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
x   = np.arange(len(splits))
w   = 0.25
for i, cls in enumerate(SEVERITY_CLASSES):
    vals = [sev_counts[s][cls] for s in splits]
    ax.bar(x + (i-1)*w, vals, w, label=cls, color=sev_colors[cls])
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_title('Severity: Crop Distribution', fontweight='bold')
ax.set_ylabel('Crop count')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Dataset Split Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Class balance check
test_sev = sev_counts['test']
sev_total = sum(test_sev.values())
print('\nSeverity test-set class balance:')
for cls in SEVERITY_CLASSES:
    pct = test_sev[cls] / sev_total * 100 if sev_total else 0
    bar = '█' * int(pct / 3)
    print(f'  {cls:<8}: {test_sev[cls]:>4} ({pct:4.1f}%)  {bar}')

imbalance = max(test_sev.values()) / max(min(test_sev.values()), 1)
if imbalance > 3:
    print(f'\n  Imbalance ratio {imbalance:.1f}x — interpret per-class F1, not overall accuracy')

In [ ]:
# ── 1d. Sample images from each split ───────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(14, 9))

for row, sp in enumerate(splits):
    img_dir = DETECTOR_DATA / 'images' / sp
    imgs    = sorted(img_dir.glob('*'))[:4] if img_dir.exists() else []
    for col in range(4):
        ax = axes[row][col]
        if col < len(imgs):
            frame = cv2.cvtColor(cv2.imread(str(imgs[col])), cv2.COLOR_BGR2RGB)
            ax.imshow(frame)
            ax.set_title(imgs[col].name[:25], fontsize=6)
        ax.axis('off')
    axes[row][0].set_ylabel(sp.upper(), fontsize=10, fontweight='bold')

plt.suptitle('Sample Images — Train / Val / Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 2 — Stage 1 Detector Validation (Test Set)

All metrics computed on the **held-out test split** (10% of data, never seen during training).

In [ ]:
# ── 2a. Load model and run test-set evaluation ──────────────────────
assert DETECTOR_MODEL.exists(), f'Missing {DETECTOR_MODEL} — run notebook 02 first'

det_model = YOLO(str(DETECTOR_MODEL))

det_metrics = det_model.val(
    data   = str(DETECTOR_DATA / 'dataset.yaml'),
    split  = 'test',
    verbose= False,
)

map50    = det_metrics.box.map50
map5095  = det_metrics.box.map
prec     = det_metrics.box.mp
rec      = det_metrics.box.mr
ap50     = det_metrics.box.ap50   # per-class AP@0.5

print('Stage 1 Detector — Test Set Results')
print('=' * 42)
print(f'  mAP@0.5       : {map50:.4f}  ({map50*100:.1f}%)')
print(f'  mAP@0.5:0.95  : {map5095:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print(f'  F1 (harmonic) : {2*prec*rec/(prec+rec+1e-9):.4f}')
print()
for i, cls in enumerate(DETECTOR_CLASSES):
    print(f'  AP@0.5 [{cls}] : {ap50[i]:.4f}')

# Qualitative label
print()
label = ('Excellent' if map50 >= 0.75 else
         'Good'      if map50 >= 0.60 else
         'Acceptable'if map50 >= 0.45 else 'Needs improvement')
print(f'  Assessment    : {label}')

In [ ]:
# ── 2b. Per-class AP bar chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
colors  = ['#e74c3c', '#2ecc71']
bars    = ax.bar(DETECTOR_CLASSES, ap50, color=colors, width=0.4, edgecolor='white', linewidth=1.5)

for bar, v in zip(bars, ap50):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)

ax.axhline(map50, color='gold', linestyle='--', linewidth=1.5, label=f'mAP@0.5 = {map50:.3f}')
ax.set_ylim(0, 1.15)
ax.set_title('Per-Class AP@0.5 — Test Set', fontweight='bold')
ax.set_ylabel('Average Precision @ IoU=0.5')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Imbalance warning
if len(ap50) >= 2 and abs(ap50[0] - ap50[1]) > 0.15:
    weaker = DETECTOR_CLASSES[int(np.argmin(ap50))]
    ax.annotate(f'← {weaker}\n  needs more data',
                xy=(int(np.argmin(ap50)), ap50.min()),
                xytext=(int(np.argmin(ap50)) + 0.15, ap50.min() + 0.1),
                arrowprops=dict(arrowstyle='->', color='red'),
                color='red', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 2c. Precision-Recall Curves ──────────────────────────────────────
# YOLO saves PR curve data in runs/detect/<run>/
# We also compute it manually from test predictions for a clean plot.

test_img_dir = DETECTOR_DATA / 'images' / 'test'
test_imgs    = sorted(test_img_dir.glob('*'))[:200]  # cap at 200 for speed

# Collect predictions at many confidence thresholds
confs   = np.linspace(0.01, 0.99, 50)
cls_data = {c: {'tp': [], 'fp': [], 'fn': []} for c in range(2)}

# Load ground truth
gt_per_img = {}
for img_path in test_imgs:
    lbl = DETECTOR_DATA / 'labels' / 'test' / (img_path.stem + '.txt')
    boxes = []
    if lbl.exists():
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                parts = line.split()
                boxes.append(int(parts[0]))
    gt_per_img[img_path] = boxes

# We use YOLO's built-in PR data from the latest run folder if available
run_dir = None
for candidate in sorted((BASE_DIR / 'runs').glob('**/results.csv'), key=lambda p: p.stat().st_mtime, reverse=True):
    run_dir = candidate.parent
    break

pr_img = None
if run_dir:
    for name in ['PR_curve.png', 'P_curve.png', 'R_curve.png']:
        f = run_dir / name
        if f.exists():
            if pr_img is None:
                pr_img = {}
            pr_img[name] = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)

if pr_img:
    keys   = list(pr_img.keys())
    fig, axes = plt.subplots(1, len(keys), figsize=(6*len(keys), 5))
    if len(keys) == 1: axes = [axes]
    titles = {'PR_curve.png': 'Precision-Recall Curve',
              'P_curve.png':  'Precision vs Confidence',
              'R_curve.png':  'Recall vs Confidence'}
    for ax, k in zip(axes, keys):
        ax.imshow(pr_img[k]); ax.axis('off')
        ax.set_title(titles.get(k, k), fontweight='bold')
    plt.suptitle('Stage 1 Detector — Precision / Recall Curves', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('PR curve PNGs not found in runs/. They are generated automatically during training.')
    print('Re-run notebook 02 Section 3 and they will appear.')

In [ ]:
# ── 2d. F1 vs Confidence Threshold ──────────────────────────────────
# Use YOLO's saved F1_curve.png if available, else compute manually

f1_img_path = None
if run_dir:
    f1_img_path = run_dir / 'F1_curve.png'

if f1_img_path and f1_img_path.exists():
    img = cv2.cvtColor(cv2.imread(str(f1_img_path)), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img); ax.axis('off')
    ax.set_title('F1 vs Confidence Threshold (from training run)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('Gold dashed line = best F1 threshold')
    print('Use this to tune configs/pipeline_config.yaml → detector_conf')
else:
    # Manual computation at fixed thresholds on test set
    print('Computing F1 vs confidence manually on test images...')
    thresholds = np.linspace(0.1, 0.9, 17)
    f1_vals    = []
    n_sample   = min(50, len(test_imgs))
    sample_imgs = list(test_imgs)[:n_sample]

    for thr in thresholds:
        tp = fp = fn = 0
        for img_path in sample_imgs:
            preds = det_model(img_path, conf=thr, verbose=False)
            gt    = gt_per_img.get(img_path, [])
            pred_cls = [int(b.cls[0]) for b in preds[0].boxes]
            for c in range(2):
                tp += min(pred_cls.count(c), gt.count(c))
                fp += max(0, pred_cls.count(c) - gt.count(c))
                fn += max(0, gt.count(c) - pred_cls.count(c))
        prec_t = tp / (tp + fp + 1e-9)
        rec_t  = tp / (tp + fn + 1e-9)
        f1_vals.append(2 * prec_t * rec_t / (prec_t + rec_t + 1e-9))

    best_idx  = int(np.argmax(f1_vals))
    best_thr  = thresholds[best_idx]
    best_f1   = f1_vals[best_idx]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thresholds, f1_vals, 'b-o', linewidth=2, markersize=4)
    ax.axvline(best_thr, color='gold', linestyle='--', linewidth=2,
               label=f'Best conf={best_thr:.2f}  F1={best_f1:.3f}')
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('F1 Score')
    ax.set_title('F1 vs Confidence Threshold — Test Set', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'\nRecommended conf threshold: {best_thr:.2f}  (current in pipeline_config: 0.40)')

In [ ]:
# ── 2e. Confusion Matrix (normalized) ───────────────────────────────
# Look for confusion_matrix_normalized.png from the training run
cm_paths = []
if run_dir:
    cm_paths = list(run_dir.glob('confusion_matrix_normalized.png'))
    if not cm_paths:
        cm_paths = list(run_dir.glob('confusion_matrix.png'))

if cm_paths:
    img = cv2.cvtColor(cv2.imread(str(cm_paths[0])), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Stage 1 Detector — Normalized Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('Rows = Actual class, Columns = Predicted class')
    print('Background row = false negatives (missed objects)')
    print('Background col = false positives (ghost detections)')
else:
    print('No confusion matrix PNG found.')
    print('Run model.val() from notebook 02 to generate it.')

In [ ]:
# ── 2f. False Positive / False Negative Examples ────────────────────
CONF_VIZ = 0.40
IOU_MATCH = 0.45

def iou(b1, b2):
    xa = max(b1[0], b2[0]); ya = max(b1[1], b2[1])
    xb = min(b1[2], b2[2]); yb = min(b1[3], b2[3])
    inter = max(0, xb-xa) * max(0, yb-ya)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-9)

fp_examples = []  # (img, frame, preds, gt_boxes, type)
fn_examples = []
tp_examples = []

cls_colors = {0: (230, 70, 70), 1: (70, 220, 70)}

for img_path in list(test_imgs)[:80]:
    frame    = cv2.imread(str(img_path))
    if frame is None: continue
    h, w     = frame.shape[:2]

    # Load GT in pixel coords
    lbl_path = DETECTOR_DATA / 'labels' / 'test' / (img_path.stem + '.txt')
    gt_boxes = []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            if not line.strip(): continue
            c, cx, cy, bw, bh = [float(x) for x in line.split()]
            x1 = (cx-bw/2)*w; y1 = (cy-bh/2)*h
            x2 = (cx+bw/2)*w; y2 = (cy+bh/2)*h
            gt_boxes.append((int(c), x1, y1, x2, y2))

    preds = det_model(img_path, conf=CONF_VIZ, verbose=False)[0].boxes
    pred_boxes = [(int(b.cls[0]), *[float(v) for v in b.xyxy[0]], float(b.conf[0]))
                  for b in preds]

    matched_gt  = set()
    matched_pred = set()
    for pi, (pc, px1,py1,px2,py2, pconf) in enumerate(pred_boxes):
        for gi, (gc, gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
            if pc == gc and iou((px1,py1,px2,py2),(gx1,gy1,gx2,gy2)) >= IOU_MATCH:
                matched_gt.add(gi); matched_pred.add(pi)

    has_tp = len(matched_pred) > 0
    has_fp = any(i not in matched_pred for i in range(len(pred_boxes)))
    has_fn = any(i not in matched_gt   for i in range(len(gt_boxes)))

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if has_tp and len(tp_examples) < 3:
        tp_examples.append((rgb, pred_boxes, gt_boxes, matched_pred, matched_gt))
    if has_fp and not has_fn and len(fp_examples) < 3:
        fp_examples.append((rgb, pred_boxes, gt_boxes, matched_pred, matched_gt))
    if has_fn and not has_fp and len(fn_examples) < 3:
        fn_examples.append((rgb, pred_boxes, gt_boxes, matched_pred, matched_gt))

    if len(tp_examples) == 3 and len(fp_examples) == 3 and len(fn_examples) == 3:
        break

def draw_boxes_on_ax(ax, rgb, pred_boxes, gt_boxes, matched_pred, matched_gt, title):
    import copy
    vis = copy.deepcopy(rgb)
    for gi, (gc, gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
        clr  = (0, 200, 0) if gi in matched_gt else (255, 165, 0)
        label = 'GT-matched' if gi in matched_gt else 'GT-missed (FN)'
        ax.add_patch(mpatches.Rectangle((gx1,gy1), gx2-gx1, gy2-gy1,
                                         linewidth=2, edgecolor=np.array(clr)/255,
                                         facecolor='none', linestyle='--'))
    for pi, (pc, px1,py1,px2,py2, pconf) in enumerate(pred_boxes):
        clr = (70, 220, 70) if pi in matched_pred else (220, 70, 70)
        tag = DETECTOR_CLASSES[pc] + ('' if pi in matched_pred else ' FP')
        ax.add_patch(mpatches.Rectangle((px1,py1), px2-px1, py2-py1,
                                         linewidth=2, edgecolor=np.array(clr)/255,
                                         facecolor='none'))
        ax.text(px1, max(py1-4,0), f'{tag} {pconf:.0%}',
                color=np.array(clr)/255, fontsize=7, fontweight='bold')
    ax.imshow(vis); ax.axis('off'); ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
row_labels = ['True Positives (correct detections)',
              'False Positives (ghost detections)',
              'False Negatives (missed objects)']
example_groups = [tp_examples, fp_examples, fn_examples]

for row, (label, group) in enumerate(zip(row_labels, example_groups)):
    for col in range(3):
        ax = axes[row][col]
        if col < len(group):
            rgb, pred_boxes, gt_boxes, matched_pred, matched_gt = group[col]
            draw_boxes_on_ax(ax, rgb, pred_boxes, gt_boxes, matched_pred, matched_gt,
                             f'{label.split("(")[0].strip()} #{col+1}')
        else:
            ax.text(0.5, 0.5, 'No example\nfound', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')
            ax.axis('off')
    axes[row][0].set_ylabel(label, fontsize=8, fontweight='bold')

plt.suptitle(f'Detection Examples — Test Set  (conf≥{CONF_VIZ}, IoU≥{IOU_MATCH})\n'
             'Solid=Predicted  Dashed=GT  Green=Matched  Red=FP  Orange=FN',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3 — Stage 2a Severity CNN Validation (Test Set)

In [ ]:
# ── 3a. Load severity model and collect test predictions ─────────────
assert SEVERITY_MODEL.exists(), f'Missing {SEVERITY_MODEL} — run notebook 03 first'

sev_model  = YOLO(str(SEVERITY_MODEL))
BATCH_SIZE = 64

test_imgs_sev, y_true_sev = [], []
for cls in SEVERITY_CLASSES:
    d = SEVERITY_DATA / 'test' / cls
    if not d.exists(): continue
    for p in sorted(d.glob('*')):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
        img = cv2.imread(str(p))
        if img is not None:
            test_imgs_sev.append(img)
            y_true_sev.append(cls)

# Batched inference
y_pred_cnn = []
y_prob_cnn = []  # softmax probabilities for ROC
for i in range(0, len(test_imgs_sev), BATCH_SIZE):
    batch = test_imgs_sev[i:i+BATCH_SIZE]
    results = sev_model(batch, imgsz=128, verbose=False)
    for r in results:
        y_pred_cnn.append(CNN_IDX_TO_LABEL[int(r.probs.top1)])
        probs = r.probs.data.cpu().numpy()   # shape (3,) in alphabetical order: High, Low, Med
        # Reorder to Low, Medium, High
        y_prob_cnn.append([probs[1], probs[2], probs[0]])

y_prob_cnn = np.array(y_prob_cnn)  # shape (N, 3)
cnn_acc    = sum(t == p for t, p in zip(y_true_sev, y_pred_cnn)) / len(y_true_sev)

print(f'Severity CNN — Test Set  ({len(y_true_sev)} crops)')
print(f'  Accuracy : {cnn_acc:.4f} ({cnn_acc*100:.1f}%)')

In [ ]:
# ── 3b. Confusion matrices (raw + normalized) ───────────────────────
cm_raw  = confusion_matrix(y_true_sev, y_pred_cnn, labels=SEVERITY_CLASSES)
cm_norm = confusion_matrix(y_true_sev, y_pred_cnn, labels=SEVERITY_CLASSES, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm_raw,  display_labels=SEVERITY_CLASSES).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Raw Counts  (acc={cnn_acc:.2%})', fontweight='bold')

ConfusionMatrixDisplay(cm_norm, display_labels=SEVERITY_CLASSES).plot(
    ax=axes[1], colorbar=False, cmap='Blues',
    values_format='.2f')
axes[1].set_title('Normalized (row = true class)', fontweight='bold')

plt.suptitle('Stage 2a Severity CNN — Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Dangerous misclassification check (High → Low)
high_idx = SEVERITY_CLASSES.index('High')
low_idx  = SEVERITY_CLASSES.index('Low')
h2l = cm_raw[high_idx][low_idx]
if h2l > 0:
    total_high = cm_raw[high_idx].sum()
    print(f'\nWARNING: {h2l}/{total_high} ({h2l/total_high:.0%}) High-severity potholes '
          f'predicted as Low (under-detection of severe hazards)')

In [ ]:
# ── 3c. Per-class precision / recall / F1 ───────────────────────────
print('Classification Report — Severity CNN — Test Set')
print('=' * 60)
print(classification_report(y_true_sev, y_pred_cnn, target_names=SEVERITY_CLASSES))

f1s = f1_score(y_true_sev, y_pred_cnn, labels=SEVERITY_CLASSES, average=None)

fig, ax = plt.subplots(figsize=(7, 4))
colors  = [sev_colors[c] for c in SEVERITY_CLASSES]
bars    = ax.bar(SEVERITY_CLASSES, f1s, color=colors, width=0.4, edgecolor='white')
for bar, v in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            f'{v:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='F1=0.5 baseline')
ax.axhline(np.mean(f1s), color='gold', linestyle='--',
           label=f'macro-avg F1 = {np.mean(f1s):.3f}')
ax.set_title('Per-Class F1 — Severity CNN — Test Set', fontweight='bold')
ax.set_ylabel('F1 Score')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3d. ROC Curves (one-vs-rest, multi-class) ───────────────────────
y_bin = label_binarize(y_true_sev, classes=SEVERITY_CLASSES)  # (N, 3)

fig, ax = plt.subplots(figsize=(7, 6))
roc_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}

aucs = {}
for i, cls in enumerate(SEVERITY_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_cnn[:, i])
    roc_auc     = auc(fpr, tpr)
    aucs[cls]   = roc_auc
    ax.plot(fpr, tpr, linewidth=2, color=roc_colors[cls],
            label=f'{cls}  (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Severity CNN (One-vs-Rest) — Test Set', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'Mean AUC: {np.mean(list(aucs.values())):.3f}')
for cls, a in aucs.items():
    rating = 'Excellent' if a >= 0.90 else 'Good' if a >= 0.80 else 'Fair' if a >= 0.70 else 'Poor'
    print(f'  {cls:<8}: AUC = {a:.3f}  ({rating})')

In [ ]:
# ── 3e. Confidence distribution per class ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (i, cls) in zip(axes, enumerate(SEVERITY_CLASSES)):
    # Confidence of the predicted class
    confs_cls = [y_prob_cnn[j, i] for j in range(len(y_true_sev)) if y_true_sev[j] == cls]
    correct_c = [y_prob_cnn[j, i] for j in range(len(y_true_sev))
                 if y_true_sev[j] == cls and y_pred_cnn[j] == cls]
    wrong_c   = [y_prob_cnn[j, i] for j in range(len(y_true_sev))
                 if y_true_sev[j] == cls and y_pred_cnn[j] != cls]
    ax.hist(correct_c, bins=20, alpha=0.7, color=roc_colors[cls], label='Correct')
    ax.hist(wrong_c,   bins=20, alpha=0.7, color='gray',           label='Misclassified')
    ax.set_title(f'{cls} Severity\n(n={len(confs_cls)})', fontweight='bold', color=roc_colors[cls])
    ax.set_xlabel(f'Predicted P({cls})')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Confidence Distribution per True Class — Severity CNN', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('High confidence + correct = well-calibrated model')
print('Low confidence + wrong    = uncertain predictions worth investigating')

In [ ]:
# ── 3f. Misclassified crop examples ─────────────────────────────────
misclassed = [(test_imgs_sev[i], y_true_sev[i], y_pred_cnn[i], y_prob_cnn[i])
              for i in range(len(y_true_sev)) if y_true_sev[i] != y_pred_cnn[i]][:12]

if misclassed:
    n   = len(misclassed)
    cols = min(n, 6)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, (img, true, pred, probs) in zip(axes_flat, misclassed):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(
            f'True: {true}\nPred: {pred}\n'
            f'L:{probs[0]:.2f} M:{probs[1]:.2f} H:{probs[2]:.2f}',
            fontsize=8,
            color='red'
        )
    for ax in axes_flat[len(misclassed):]:
        ax.axis('off')

    plt.suptitle('Misclassified Severity Crops (True → Predicted)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No misclassified examples in test set — perfect accuracy!')

---
## Section 4 — Stage 2a Severity: Classical CV Baseline

Evaluating the hand-crafted feature method on the same test set for direct comparison.

In [ ]:
from analyzers.pothole_analyzer import _compute_cv_scores, _cv_severity
import time

t0 = time.time()
y_pred_cv  = []
cv_scores  = []
for img in test_imgs_sev:
    sc = _compute_cv_scores(img)
    cv_scores.append(sc)
    y_pred_cv.append(_cv_severity(sc))
cv_time = time.time() - t0

cv_acc = sum(t == p for t, p in zip(y_true_sev, y_pred_cv)) / len(y_true_sev)
print(f'Classical CV — Test Set  ({len(y_true_sev)} crops)')
print(f'  Accuracy : {cv_acc:.4f} ({cv_acc*100:.1f}%)')
print(f'  Speed    : {len(y_true_sev)/cv_time:.0f} crops/sec')
print()
print(classification_report(y_true_sev, y_pred_cv, target_names=SEVERITY_CLASSES))

In [ ]:
# ── 4a. Side-by-side confusion matrices ─────────────────────────────
cm_cv   = confusion_matrix(y_true_sev, y_pred_cv,  labels=SEVERITY_CLASSES, normalize='true')
cm_cnn  = confusion_matrix(y_true_sev, y_pred_cnn, labels=SEVERITY_CLASSES, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_cv,  display_labels=SEVERITY_CLASSES).plot(
    ax=axes[0], colorbar=False, cmap='Blues', values_format='.2f')
axes[0].set_title(f'Classical CV  (acc={cv_acc:.2%})', fontweight='bold')

ConfusionMatrixDisplay(cm_cnn, display_labels=SEVERITY_CLASSES).plot(
    ax=axes[1], colorbar=False, cmap='Oranges', values_format='.2f')
axes[1].set_title(f'Fine-tuned CNN  (acc={cnn_acc:.2%})', fontweight='bold')

plt.suptitle('Normalized Confusion Matrices — CV vs CNN Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 4b. Per-class F1 grouped bar ─────────────────────────────────────
cv_f1s  = f1_score(y_true_sev, y_pred_cv,  labels=SEVERITY_CLASSES, average=None)
cnn_f1s = f1_score(y_true_sev, y_pred_cnn, labels=SEVERITY_CLASSES, average=None)
x       = np.arange(len(SEVERITY_CLASSES))
w       = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# F1 comparison
axes[0].bar(x - w/2, cv_f1s,  w, label='Classical CV',    color='#3498db', alpha=0.85)
axes[0].bar(x + w/2, cnn_f1s, w, label='Fine-tuned CNN',  color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(SEVERITY_CLASSES)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1: CV vs CNN', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')
for xi, (c, n) in enumerate(zip(cv_f1s, cnn_f1s)):
    axes[0].text(xi-w/2, c+0.01, f'{c:.2f}', ha='center', fontsize=9)
    axes[0].text(xi+w/2, n+0.01, f'{n:.2f}', ha='center', fontsize=9)

# Accuracy + speed summary
methods = ['Classical CV', 'Fine-tuned CNN']
accs    = [cv_acc, cnn_acc]
winner  = 'CNN' if cnn_acc >= cv_acc else 'CV'
axes[1].bar(methods, accs, color=['#3498db','#e74c3c'], width=0.4, alpha=0.85)
for i, (m, a) in enumerate(zip(methods, accs)):
    axes[1].text(i, a + 0.01, f'{a:.2%}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title(f'Overall Accuracy — Winner: {winner}', fontweight='bold')
axes[1].axhline(1/3, color='gray', linestyle=':', alpha=0.5, label='Random baseline (33%)')
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Severity Method Comparison — CV vs CNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 5 — End-to-End Pipeline Validation

In [ ]:
import time
from pipeline import Pipeline

pipe = Pipeline()

val_dir   = DETECTOR_DATA / 'images' / 'val'
val_imgs  = sorted(val_dir.glob('*'))[:50]
frames    = [cv2.imread(str(p)) for p in val_imgs if cv2.imread(str(p)) is not None]

# Warm-up
for f in frames[:3]:
    pipe.run(f)
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Benchmark
t0 = time.time()
for f in frames:
    pipe.run(f)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.time() - t0

fps = len(frames) / elapsed
ms  = elapsed / len(frames) * 1000

print(f'Pipeline Speed Benchmark  ({len(frames)} frames)')
print('=' * 42)
print(f'  Per frame : {ms:.1f} ms')
print(f'  FPS       : {fps:.1f}')
print()
if fps >= 25:
    rt_label = 'Real-time capable (≥25 FPS)'
elif fps >= 15:
    rt_label = 'Near real-time (15–25 FPS)'
else:
    rt_label = f'Below real-time ({fps:.0f} FPS) — consider YOLOv8n or lower resolution'
print(f'  Assessment: {rt_label}')

In [ ]:
# ── 5a. Detection statistics over validation set ─────────────────────
ph_detections  = []
tl_detections  = []
severity_counts = Counter()
tl_color_counts = Counter()

for f in frames[:80]:
    r = pipe.run(f)
    ph_detections.append(len(r['potholes']))
    tl_detections.append(len(r['traffic_lights']))
    for ph in r['potholes']:
        severity_counts[ph['severity']] += 1
    for tl in r['traffic_lights']:
        tl_color_counts[tl['color']] += 1

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Detections per frame histogram
axes[0].hist(ph_detections, bins=range(0, max(ph_detections)+2), color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Potholes per frame'); axes[0].set_ylabel('Frame count')
axes[0].set_title(f'Pothole Detections\nMean={np.mean(ph_detections):.2f}', fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].hist(tl_detections, bins=range(0, max(tl_detections)+2), color='#2ecc71', alpha=0.8)
axes[1].set_xlabel('Traffic lights per frame'); axes[1].set_ylabel('Frame count')
axes[1].set_title(f'Traffic Light Detections\nMean={np.mean(tl_detections):.2f}', fontweight='bold')
axes[1].grid(alpha=0.3)

# Severity distribution
sev_labels = [c for c in SEVERITY_CLASSES if c in severity_counts]
sev_vals   = [severity_counts[c] for c in sev_labels]
axes[2].bar(sev_labels, sev_vals, color=[sev_colors[c] for c in sev_labels], width=0.5)
for i, (l, v) in enumerate(zip(sev_labels, sev_vals)):
    axes[2].text(i, v+0.5, str(v), ha='center', fontweight='bold')
axes[2].set_title('Predicted Severity Distribution', fontweight='bold')
axes[2].grid(alpha=0.3, axis='y')

# Traffic light color distribution
tl_labels = list(tl_color_counts.keys())
tl_vals   = [tl_color_counts[k] for k in tl_labels]
tl_clrs   = {'Red': '#e74c3c', 'Yellow': '#f1c40f', 'Green': '#2ecc71', 'Unknown': 'gray'}
axes[3].bar(tl_labels, tl_vals,
            color=[tl_clrs.get(k,'steelblue') for k in tl_labels], width=0.5)
for i, (l, v) in enumerate(zip(tl_labels, tl_vals)):
    axes[3].text(i, v+0.5, str(v), ha='center', fontweight='bold')
axes[3].set_title('Traffic Light Color Distribution', fontweight='bold')
axes[3].grid(alpha=0.3, axis='y')

plt.suptitle(f'Pipeline Output Statistics — {len(frames)} Validation Frames',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Annotated output grid (3×3) ─────────────────────────────────
grid_frames = [cv2.imread(str(p)) for p in val_imgs[:9]]
fig, axes   = plt.subplots(3, 3, figsize=(15, 10))

for ax, f in zip(axes.flatten(), grid_frames):
    if f is None:
        ax.axis('off'); continue
    r   = pipe.run(f)
    ann = cv2.cvtColor(r['annotated_frame'], cv2.COLOR_BGR2RGB)
    ax.imshow(ann); ax.axis('off')
    ph_n = len(r['potholes'])
    tl_n = len(r['traffic_lights'])
    sev  = r['potholes'][0]['severity']   if r['potholes']       else ''
    clr  = r['traffic_lights'][0]['color'] if r['traffic_lights'] else ''
    ax.set_title(f'PH:{ph_n}{" "+sev if sev else ""}  TL:{tl_n}{" "+clr if clr else ""}',
                 fontsize=8)

plt.suptitle('End-to-End Pipeline — Annotated Outputs (Validation Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 6 — Validation Summary Dashboard

All key metrics in a single view.

In [ ]:
# ── 6a. Metrics table ────────────────────────────────────────────────
import pandas as pd

rows = [
    ['Stage 1 Detector', 'mAP@0.5',          f'{map50:.4f}'],
    ['Stage 1 Detector', 'mAP@0.5:0.95',     f'{map5095:.4f}'],
    ['Stage 1 Detector', 'Precision',         f'{prec:.4f}'],
    ['Stage 1 Detector', 'Recall',            f'{rec:.4f}'],
    ['Stage 1 Detector', f'AP@0.5 ({DETECTOR_CLASSES[0]})',   f'{ap50[0]:.4f}'],
    ['Stage 1 Detector', f'AP@0.5 ({DETECTOR_CLASSES[1]})',   f'{ap50[1]:.4f}'],
    ['Stage 2a CNN',     'Test Accuracy',     f'{cnn_acc:.4f}'],
    ['Stage 2a CNN',     'Macro F1',          f'{np.mean(cnn_f1s):.4f}'],
    ['Stage 2a CNN',     'F1 Low',            f'{cnn_f1s[0]:.4f}'],
    ['Stage 2a CNN',     'F1 Medium',         f'{cnn_f1s[1]:.4f}'],
    ['Stage 2a CNN',     'F1 High',           f'{cnn_f1s[2]:.4f}'],
    ['Stage 2a CNN',     'Mean ROC-AUC',      f"{np.mean(list(aucs.values())):.4f}"],
    ['Stage 2a CV',      'Test Accuracy',     f'{cv_acc:.4f}'],
    ['Stage 2a CV',      'Macro F1',          f'{np.mean(cv_f1s):.4f}'],
    ['Pipeline',         'FPS (val frames)',  f'{fps:.1f}'],
    ['Pipeline',         'ms per frame',      f'{ms:.1f}'],
]

df_summary = pd.DataFrame(rows, columns=['Stage', 'Metric', 'Value'])
print(df_summary.to_string(index=False))

In [ ]:
# ── 6b. Summary visual dashboard ────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.4)

# -- Detector mAP gauge (top-left 2x1)
ax1 = fig.add_subplot(gs[0, 0])
theta   = np.linspace(np.pi, 0, 200)
ax1.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=8)
filled  = np.linspace(np.pi, np.pi*(1 - map50), 200)
color   = '#2ecc71' if map50 >= 0.70 else '#f39c12' if map50 >= 0.50 else '#e74c3c'
ax1.plot(np.cos(filled), np.sin(filled), color, linewidth=8)
ax1.text(0, 0.1, f'{map50:.3f}', ha='center', va='center', fontsize=20, fontweight='bold')
ax1.text(0, -0.25, 'mAP@0.5', ha='center', fontsize=10)
ax1.set_xlim(-1.3, 1.3); ax1.set_ylim(-0.5, 1.2)
ax1.axis('off'); ax1.set_title('Stage 1 Detector', fontweight='bold')

# -- Severity CNN accuracy gauge
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=8)
filled2 = np.linspace(np.pi, np.pi*(1 - cnn_acc), 200)
color2  = '#2ecc71' if cnn_acc >= 0.70 else '#f39c12' if cnn_acc >= 0.55 else '#e74c3c'
ax2.plot(np.cos(filled2), np.sin(filled2), color2, linewidth=8)
ax2.text(0, 0.1, f'{cnn_acc:.3f}', ha='center', va='center', fontsize=20, fontweight='bold')
ax2.text(0, -0.25, 'Accuracy', ha='center', fontsize=10)
ax2.set_xlim(-1.3, 1.3); ax2.set_ylim(-0.5, 1.2)
ax2.axis('off'); ax2.set_title('Stage 2a CNN (Severity)', fontweight='bold')

# -- ROC-AUC bar
ax3 = fig.add_subplot(gs[0, 2])
auc_classes = list(aucs.keys())
auc_vals    = [aucs[c] for c in auc_classes]
bars3 = ax3.bar(auc_classes, auc_vals, color=[roc_colors[c] for c in auc_classes], width=0.5)
for b, v in zip(bars3, auc_vals):
    ax3.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
ax3.set_ylim(0, 1.15); ax3.set_ylabel('AUC')
ax3.set_title('ROC-AUC per Severity Class', fontweight='bold')
ax3.axhline(0.5, linestyle=':', color='gray', alpha=0.5)
ax3.grid(alpha=0.3, axis='y')

# -- CV vs CNN accuracy
ax4 = fig.add_subplot(gs[0, 3])
ax4.bar(['CV', 'CNN'], [cv_acc, cnn_acc], color=['#3498db', '#e74c3c'], width=0.4)
for i, (l, v) in enumerate([('CV', cv_acc), ('CNN', cnn_acc)]):
    ax4.text(i, v+0.01, f'{v:.2%}', ha='center', fontweight='bold')
ax4.set_ylim(0, 1.15); ax4.set_ylabel('Accuracy')
ax4.set_title('Severity Method Comparison', fontweight='bold')
ax4.axhline(1/3, linestyle=':', color='gray', alpha=0.5, label='Random')
ax4.legend(fontsize=8); ax4.grid(alpha=0.3, axis='y')

# -- Per-class F1 comparison (bottom left 2 cols)
ax5 = fig.add_subplot(gs[1, :2])
x   = np.arange(len(SEVERITY_CLASSES))
w   = 0.3
ax5.bar(x - w/2, cv_f1s,  w, label='CV',  color='#3498db', alpha=0.85)
ax5.bar(x + w/2, cnn_f1s, w, label='CNN', color='#e74c3c', alpha=0.85)
ax5.set_xticks(x); ax5.set_xticklabels(SEVERITY_CLASSES)
ax5.set_ylim(0, 1.15); ax5.set_ylabel('F1 Score')
ax5.set_title('Per-Class F1: CV vs CNN', fontweight='bold')
ax5.legend(); ax5.grid(alpha=0.3, axis='y')

# -- Metrics table (bottom right 2 cols)
ax6 = fig.add_subplot(gs[1, 2:])
ax6.axis('off')
key_metrics = [
    ['Metric',                'Value',          'Rating'],
    ['Detector mAP@0.5',      f'{map50:.3f}',   'Excellent' if map50>=0.75 else 'Good' if map50>=0.60 else 'Fair'],
    ['Detector mAP@0.5:0.95', f'{map5095:.3f}', '—'],
    ['Severity CNN Acc',      f'{cnn_acc:.3f}', 'Good' if cnn_acc>=0.70 else 'Fair'],
    ['Severity CV Acc',       f'{cv_acc:.3f}',  'Good' if cv_acc>=0.70 else 'Fair'],
    ['Mean ROC-AUC',          f"{np.mean(list(aucs.values())):.3f}", '—'],
    ['Pipeline FPS',          f'{fps:.1f}',     'Real-time' if fps>=25 else 'Near RT' if fps>=15 else 'Slow'],
]
tbl = ax6.table(cellText=key_metrics[1:], colLabels=key_metrics[0],
                cellLoc='center', loc='center', bbox=[0.05, 0.05, 0.9, 0.9])
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#ecf0f1')
ax6.set_title('Key Metrics Summary', fontweight='bold', pad=8)

plt.suptitle('Smart Road Assistant — Validation Summary Dashboard',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig(str(BASE_DIR / 'validation_summary.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved → validation_summary.png')

In [ ]:
# ── 6c. Print final assessment ───────────────────────────────────────
print('=' * 60)
print('  SMART ROAD ASSISTANT — FINAL VALIDATION REPORT')
print('=' * 60)
print()
print(f'  Stage 1 — Object Detector (YOLOv8s)')
print(f'    mAP@0.5        : {map50:.4f}  ({"Excellent" if map50>=0.75 else "Good" if map50>=0.60 else "Fair"})')
print(f'    mAP@0.5:0.95   : {map5095:.4f}')
print(f'    Precision      : {prec:.4f}')
print(f'    Recall         : {rec:.4f}')
print(f'    AP pothole     : {ap50[0]:.4f}')
print(f'    AP traffic_light: {ap50[1]:.4f}')
print()
print(f'  Stage 2a — Severity Classifier')
print(f'    CNN accuracy   : {cnn_acc:.4f}')
print(f'    CV  accuracy   : {cv_acc:.4f}')
print(f'    Winner         : {"CNN" if cnn_acc >= cv_acc else "CV"}')
print(f'    Macro F1 (CNN) : {np.mean(cnn_f1s):.4f}')
print(f'    Mean AUC (CNN) : {np.mean(list(aucs.values())):.4f}')
print()
print(f'  End-to-end Pipeline')
print(f'    FPS            : {fps:.1f}  ({"real-time" if fps>=25 else "near real-time" if fps>=15 else "below real-time"})')
print(f'    ms per frame   : {ms:.1f}')
print('=' * 60)